In [1]:
"""Einstiegspunkt:  python run.py   (Einstellungen in src/config.py)"""
import warnings
import numpy as np
from sklearn.metrics import roc_auc_score

from src import config as cfg
from src.cv import run_cv, run_tag
from src.data import adversarial_validation, load_data
from src.ensemble import blend, rank01
from src.submission import log_experiment, write_submission

warnings.filterwarnings("ignore")

In [ ]:
def main():
    data = load_data(cfg.FEATURE_SET)
    if cfg.RUN_ADVERSARIAL:
        adversarial_validation(data.X, data.X_test)

    results = {}
    for name in cfg.MODELS:
        # Checkpoint-Prüfung auf Modell-Ebene
        tag = run_tag(name, cfg.FEATURE_SET)
        oof_file = cfg.MODELS_DIR / f"oof_{tag}.npy"
        test_file = cfg.MODELS_DIR / f"test_{tag}.npy"

        # Falls ein Modell bereits komplett fertig war, aus Dateien laden
        if oof_file.exists() and test_file.exists():
            print(f"==> [{name}] Modell bereits vollständig berechnet. Lade Ergebnisse aus Checkpoint ({oof_file.name})...")
            oof_pred = np.load(oof_file)
            test_pred = np.load(test_file)
            auc = roc_auc_score(data.y, oof_pred)
            results[name] = {"oof": oof_pred, "test": test_pred, "auc": auc, "tag": tag}
        else:
            # Ansonsten run_cv aufrufen (welches in-fold Checkpoints nutzt)
            results[name] = run_cv(name, data, cfg.FEATURE_SET)

    print(
        "Zusammenfassung:",
        {n: round(r["auc"], 6) for n, r in results.items()},
    )
    if len(results) > 1:
        # mode: "prob" (Roh-Wahrscheinlichkeiten) | "logit" | "rank"
        final_pred, _, final_auc = blend(results, data.y, mode="prob")
        print(f"--> Blend OOF-AUC: {final_auc:.6f}")
        log_experiment("blend", final_auc)
    else:
        only = next(iter(results.values()))
        final_pred, final_auc = only["test"], only["auc"]
    
    write_submission(data.test, data.sample_sub, final_pred, final_auc)

In [3]:
if __name__ == "__main__":
    main()
 

Originaldatensatz geladen: (129880, 22)
train (699635, 23) | test (299844, 22) | sample_submission (299844, 2)
Trainiere Original-Data Teacher Modell...
Teacher OOF-AUC auf Original: 0.994973
Original-Data Teacher Feature erfolgreich generiert!
Hinweis: Test hat keine fehlenden Arrival-Delay-Werte -> 'Arrival_Delay_Missing' entfernt
Feature-Set fs07: 91 Features (kategorisch: 9)
==> [xgb] Modell bereits vollständig berechnet. Lade Ergebnisse aus Checkpoint (oof_xgb_fs07_noOrigRows_s42-2026-10-23-16-2-8-46.npy)...
==> [lgbm] Modell bereits vollständig berechnet. Lade Ergebnisse aus Checkpoint (oof_lgbm_fs07_noOrigRows_s42-2026-10-23-16-2-8-46.npy)...
==> [cat] Modell bereits vollständig berechnet. Lade Ergebnisse aus Checkpoint (oof_cat_fs07_noOrigRows_s42-2026-10-23-16-2-8-46.npy)...
Zusammenfassung: {'xgb': 0.961248, 'lgbm': 0.95712, 'cat': 0.96111}
Blend [prob] (gleiche Gewichte):    0.960177
Blend [prob] (optimierte Gewichte): 0.961291  {'xgb': 0.654, 'lgbm': 0.0, 'cat': 0.346}
--> 